## STD AMBIENTE 0.003 BUFFER 0.003 ##

In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from environment import TrackingEnv
from BC_training import BCModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_policy(checkpoint_path_transl, env=None):
    if env is None:
        env = TrackingEnv()

    # stato: x, y, x_target, y_target
    state_dim_trasl = 4
    agent_transl = BCModel(state_dim_trasl, 2)

    ckpt_transl = torch.load(checkpoint_path_transl, map_location="cpu")
    agent_transl.load_state_dict(ckpt_transl["actor_state_dict"])
    agent_transl.eval()
    return agent_transl

def test_bc_policy_rot(model, num_episodes=1000, tolerance=0.02, noise_std=0.003):
    env = TrackingEnv()
    attached_steps_list = []

    for ep in range(num_episodes):
        state, _ = env.reset()
        real_state = torch.tensor(state, dtype=torch.float32).to(device)
        state = torch.tensor(state, dtype=torch.float32).to(device)
        state = real_state.clone()
        state[2:4] += torch.normal(mean=0.0, std=noise_std, size=(2,), device=state.device)

        done = False
        attached_counter = 0
        steps = 0
        agent_traj = []
        target_traj = []

        while not done:
            agent_traj.append(state[0].item())
            target_traj.append(state[1].item())

            with torch.no_grad():
                action = model(state).squeeze().cpu()

            #action = np.clip(action, env.action_space.low, env.action_space.high)
            next_state, _, done, truncated, _, _ = env.step(action)
            
            real_next_state = torch.tensor(next_state, dtype=torch.float32).to(device)
            next_state = torch.tensor(next_state, dtype=torch.float32).to(device)
            next_state = real_next_state.clone()
            
            next_state[2:4] += torch.normal(mean=0.0, std=noise_std, size=(2,), device=state.device)

            dist = torch.norm(real_next_state[:2] - real_state[2:4])
            if dist < tolerance:
                attached_counter += 1
      
            steps += 1

            real_state = real_next_state
            state = next_state
            done = truncated
            #if attached_counter > 0 and dist > tolerance:
            #    done = True
            

        attached_steps_list.append(attached_counter)
        #print(f"[Ep {ep}] Passi attaccati: {attached_counter}")

        #if ep % 20 == 0:
        #    plot_rotation_trajectory(agent_traj, target_traj, ep)

    print("\nRisultati complessivi:")
    print(f"Media passi attaccati: {np.mean(attached_steps_list):.2f}")
    print(f"Episodi con > 20 passi attaccati: {sum(c > 20 for c in attached_steps_list)} su {num_episodes}")
    print(f"Episodi con > 80 passi attaccati: {sum(c > 80 for c in attached_steps_list)} su {num_episodes}")
    print(f"Episodi con < 70 passi attaccati: {sum(c < 70 for c in attached_steps_list)} su {num_episodes}")
    env.close()

def plot_rotation_trajectory(agent_traj, target_traj, ep):
    agent_traj = np.array(agent_traj)
    target_traj = np.array(target_traj)

    plt.figure(figsize=(6, 4))
    plt.plot(agent_traj, label="Agente (theta)")
    plt.plot(target_traj, label="Target (theta)")
    plt.title(f"Rotazione - Episodio {ep}")
    plt.xlabel("Step")
    plt.ylabel("Theta")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    plt.close()

# --- MAIN ---
if __name__ == "__main__":
    bc_policy = load_policy("/Users/edoardozappia/Documents/GitHub/RL_IL_AR_for_surgery/Esperimento_1_corretto/CoL/Traslazioni-dinamiche/Combinazioni_test/ddpg_mov_0.05_std_0.003_buffer_pieno_0.003_no-init_20250724_115005/checkpoint_ep255.pth")
    test_bc_policy_rot(bc_policy, num_episodes=1000, noise_std=0.003)



Risultati complessivi:
Media passi attaccati: 95.05
Episodi con > 20 passi attaccati: 1000 su 1000
Episodi con > 80 passi attaccati: 961 su 1000
Episodi con < 70 passi attaccati: 18 su 1000


## STD AMBIENTE 0.003 BUFFER 0.005 ##

In [2]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from environment import TrackingEnv
from BC_training import BCModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_policy(checkpoint_path_transl, env=None):
    if env is None:
        env = TrackingEnv()

    # stato: x, y, x_target, y_target
    state_dim_trasl = 4
    agent_transl = BCModel(state_dim_trasl, 2)

    ckpt_transl = torch.load(checkpoint_path_transl, map_location="cpu")
    agent_transl.load_state_dict(ckpt_transl["actor_state_dict"])
    agent_transl.eval()
    return agent_transl

def test_bc_policy_rot(model, num_episodes=1000, tolerance=0.02, noise_std=0.003):
    env = TrackingEnv()
    attached_steps_list = []

    for ep in range(num_episodes):
        state, _ = env.reset()
        real_state = torch.tensor(state, dtype=torch.float32).to(device)
        state = torch.tensor(state, dtype=torch.float32).to(device)
        state = real_state.clone()
        state[2:4] += torch.normal(mean=0.0, std=noise_std, size=(2,), device=state.device)

        done = False
        attached_counter = 0
        steps = 0
        agent_traj = []
        target_traj = []

        while not done:
            agent_traj.append(state[0].item())
            target_traj.append(state[1].item())

            with torch.no_grad():
                action = model(state).squeeze().cpu()

            #action = np.clip(action, env.action_space.low, env.action_space.high)
            next_state, _, done, truncated, _, _ = env.step(action)
            
            real_next_state = torch.tensor(next_state, dtype=torch.float32).to(device)
            next_state = torch.tensor(next_state, dtype=torch.float32).to(device)
            next_state = real_next_state.clone()
            
            next_state[2:4] += torch.normal(mean=0.0, std=noise_std, size=(2,), device=state.device)

            dist = torch.norm(real_next_state[:2] - real_state[2:4])
            if dist < tolerance:
                attached_counter += 1
      
            steps += 1

            real_state = real_next_state
            state = next_state
            done = truncated
            #if attached_counter > 0 and dist > tolerance:
            #    done = True
            

        attached_steps_list.append(attached_counter)
        #print(f"[Ep {ep}] Passi attaccati: {attached_counter}")

        #if ep % 20 == 0:
        #    plot_rotation_trajectory(agent_traj, target_traj, ep)

    print("\nRisultati complessivi:")
    print(f"Media passi attaccati: {np.mean(attached_steps_list):.2f}")
    print(f"Episodi con > 20 passi attaccati: {sum(c > 20 for c in attached_steps_list)} su {num_episodes}")
    print(f"Episodi con > 80 passi attaccati: {sum(c > 80 for c in attached_steps_list)} su {num_episodes}")
    print(f"Episodi con < 70 passi attaccati: {sum(c < 70 for c in attached_steps_list)} su {num_episodes}")
    env.close()

def plot_rotation_trajectory(agent_traj, target_traj, ep):
    agent_traj = np.array(agent_traj)
    target_traj = np.array(target_traj)

    plt.figure(figsize=(6, 4))
    plt.plot(agent_traj, label="Agente (theta)")
    plt.plot(target_traj, label="Target (theta)")
    plt.title(f"Rotazione - Episodio {ep}")
    plt.xlabel("Step")
    plt.ylabel("Theta")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    plt.close()

# --- MAIN ---
if __name__ == "__main__":
    bc_policy = load_policy("/Users/edoardozappia/Documents/GitHub/RL_IL_AR_for_surgery/Esperimento_1_corretto/CoL/Traslazioni-dinamiche/Combinazioni_test/ddpg_mov_0.05_std_0.003_buffer_pieno_0.005_no-init_20250724_115232/checkpoint_ep521.pth")
    test_bc_policy_rot(bc_policy, num_episodes=1000, noise_std=0.003)



Risultati complessivi:
Media passi attaccati: 90.99
Episodi con > 20 passi attaccati: 998 su 1000
Episodi con > 80 passi attaccati: 880 su 1000
Episodi con < 70 passi attaccati: 78 su 1000


## STD AMBIENTE 0.005 BUFFER 0.003 ##

In [3]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from environment import TrackingEnv
from BC_training import BCModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_policy(checkpoint_path_transl, env=None):
    if env is None:
        env = TrackingEnv()

    # stato: x, y, x_target, y_target
    state_dim_trasl = 4
    agent_transl = BCModel(state_dim_trasl, 2)

    ckpt_transl = torch.load(checkpoint_path_transl, map_location="cpu")
    agent_transl.load_state_dict(ckpt_transl["actor_state_dict"])
    agent_transl.eval()
    return agent_transl

def test_bc_policy_rot(model, num_episodes=1000, tolerance=0.02, noise_std=0.005):
    env = TrackingEnv()
    attached_steps_list = []

    for ep in range(num_episodes):
        state, _ = env.reset()
        real_state = torch.tensor(state, dtype=torch.float32).to(device)
        state = torch.tensor(state, dtype=torch.float32).to(device)
        state = real_state.clone()
        state[2:4] += torch.normal(mean=0.0, std=noise_std, size=(2,), device=state.device)

        done = False
        attached_counter = 0
        steps = 0
        agent_traj = []
        target_traj = []

        while not done:
            agent_traj.append(state[0].item())
            target_traj.append(state[1].item())

            with torch.no_grad():
                action = model(state).squeeze().cpu()

            #action = np.clip(action, env.action_space.low, env.action_space.high)
            next_state, _, done, truncated, _, _ = env.step(action)
            
            real_next_state = torch.tensor(next_state, dtype=torch.float32).to(device)
            next_state = torch.tensor(next_state, dtype=torch.float32).to(device)
            next_state = real_next_state.clone()
            
            next_state[2:4] += torch.normal(mean=0.0, std=noise_std, size=(2,), device=state.device)

            dist = torch.norm(real_next_state[:2] - real_state[2:4])
            if dist < tolerance:
                attached_counter += 1
      
            steps += 1

            real_state = real_next_state
            state = next_state
            done = truncated
            #if attached_counter > 0 and dist > tolerance:
            #    done = True
            

        attached_steps_list.append(attached_counter)
        #print(f"[Ep {ep}] Passi attaccati: {attached_counter}")

        #if ep % 20 == 0:
        #    plot_rotation_trajectory(agent_traj, target_traj, ep)

    print("\nRisultati complessivi:")
    print(f"Media passi attaccati: {np.mean(attached_steps_list):.2f}")
    print(f"Episodi con > 20 passi attaccati: {sum(c > 20 for c in attached_steps_list)} su {num_episodes}")
    print(f"Episodi con > 80 passi attaccati: {sum(c > 80 for c in attached_steps_list)} su {num_episodes}")
    print(f"Episodi con < 70 passi attaccati: {sum(c < 70 for c in attached_steps_list)} su {num_episodes}")
    env.close()

def plot_rotation_trajectory(agent_traj, target_traj, ep):
    agent_traj = np.array(agent_traj)
    target_traj = np.array(target_traj)

    plt.figure(figsize=(6, 4))
    plt.plot(agent_traj, label="Agente (theta)")
    plt.plot(target_traj, label="Target (theta)")
    plt.title(f"Rotazione - Episodio {ep}")
    plt.xlabel("Step")
    plt.ylabel("Theta")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    plt.close()

# --- MAIN ---
if __name__ == "__main__":
    bc_policy = load_policy("/Users/edoardozappia/Documents/GitHub/RL_IL_AR_for_surgery/Esperimento_1_corretto/CoL/Traslazioni-dinamiche/Combinazioni_test/ddpg_mov_0.05_std_0.005_buffer_pieno_0.003_no-init_20250724_115934/checkpoint_ep3738.pth")
    test_bc_policy_rot(bc_policy, num_episodes=1000, noise_std=0.005)



Risultati complessivi:
Media passi attaccati: 91.87
Episodi con > 20 passi attaccati: 1000 su 1000
Episodi con > 80 passi attaccati: 922 su 1000
Episodi con < 70 passi attaccati: 35 su 1000


## STD AMBIENTE 0.005 BUFFER 0.005 ##

In [4]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from environment import TrackingEnv
from BC_training import BCModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_policy(checkpoint_path_transl, env=None):
    if env is None:
        env = TrackingEnv()

    # stato: x, y, x_target, y_target
    state_dim_trasl = 4
    agent_transl = BCModel(state_dim_trasl, 2)

    ckpt_transl = torch.load(checkpoint_path_transl, map_location="cpu")
    agent_transl.load_state_dict(ckpt_transl["actor_state_dict"])
    agent_transl.eval()
    return agent_transl

def test_bc_policy_rot(model, num_episodes=1000, tolerance=0.02, noise_std=0.005):
    env = TrackingEnv()
    attached_steps_list = []

    for ep in range(num_episodes):
        state, _ = env.reset()
        real_state = torch.tensor(state, dtype=torch.float32).to(device)
        state = torch.tensor(state, dtype=torch.float32).to(device)
        state = real_state.clone()
        state[2:4] += torch.normal(mean=0.0, std=noise_std, size=(2,), device=state.device)

        done = False
        attached_counter = 0
        steps = 0
        agent_traj = []
        target_traj = []

        while not done:
            agent_traj.append(state[0].item())
            target_traj.append(state[1].item())

            with torch.no_grad():
                action = model(state).squeeze().cpu()

            #action = np.clip(action, env.action_space.low, env.action_space.high)
            next_state, _, done, truncated, _, _ = env.step(action)
            
            real_next_state = torch.tensor(next_state, dtype=torch.float32).to(device)
            next_state = torch.tensor(next_state, dtype=torch.float32).to(device)
            next_state = real_next_state.clone()
            
            next_state[2:4] += torch.normal(mean=0.0, std=noise_std, size=(2,), device=state.device)

            dist = torch.norm(real_next_state[:2] - real_state[2:4])
            if dist < tolerance:
                attached_counter += 1
      
            steps += 1

            real_state = real_next_state
            state = next_state
            done = truncated
            #if attached_counter > 0 and dist > tolerance:
            #    done = True
            

        attached_steps_list.append(attached_counter)
        #print(f"[Ep {ep}] Passi attaccati: {attached_counter}")

        #if ep % 20 == 0:
        #    plot_rotation_trajectory(agent_traj, target_traj, ep)

    print("\nRisultati complessivi:")
    print(f"Media passi attaccati: {np.mean(attached_steps_list):.2f}")
    print(f"Episodi con > 20 passi attaccati: {sum(c > 20 for c in attached_steps_list)} su {num_episodes}")
    print(f"Episodi con > 80 passi attaccati: {sum(c > 80 for c in attached_steps_list)} su {num_episodes}")
    print(f"Episodi con < 70 passi attaccati: {sum(c < 70 for c in attached_steps_list)} su {num_episodes}")
    env.close()

def plot_rotation_trajectory(agent_traj, target_traj, ep):
    agent_traj = np.array(agent_traj)
    target_traj = np.array(target_traj)

    plt.figure(figsize=(6, 4))
    plt.plot(agent_traj, label="Agente (theta)")
    plt.plot(target_traj, label="Target (theta)")
    plt.title(f"Rotazione - Episodio {ep}")
    plt.xlabel("Step")
    plt.ylabel("Theta")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    plt.close()

# --- MAIN ---
if __name__ == "__main__":
    bc_policy = load_policy("/Users/edoardozappia/Documents/GitHub/RL_IL_AR_for_surgery/Esperimento_1_corretto/CoL/Traslazioni-dinamiche/Combinazioni_test/ddpg_mov_0.05_std_0.005_buffer_pieno_0.005_no-init_20250724_121054/checkpoint_ep1506.pth")
    test_bc_policy_rot(bc_policy, num_episodes=1000, noise_std=0.005)



Risultati complessivi:
Media passi attaccati: 90.42
Episodi con > 20 passi attaccati: 998 su 1000
Episodi con > 80 passi attaccati: 873 su 1000
Episodi con < 70 passi attaccati: 82 su 1000
